# 04 — Feature engineering for VFM

Code: `src/features.py`. Two tables built with the **same** summary function (so training = prediction):

- **test table** (85 rows): features averaged over each well-test window + measured rates (labels)
- **daily table** (2,730 rows = 10 wells × 273 days): features per calendar day → what we must predict

### Features and the physics behind them

| Feature | Formula | Why it controls the rate |
|---|---|---|
| `choke_flow_term` | `size² · √(WHP − P_down)` | Orifice equation: flow ∝ opening area × √(pressure drop) |
| `gilbert_term` | `WHP · size^1.89` | Gilbert's field correlation for critical flow through a choke |
| `choke_p_ratio` | `P_down / WHP` | < ~0.55 → critical flow (rate no longer depends on downstream pressure) |
| `tubing_dp`, `tubing_gradient` | `Pwf − WHP`, `/ depth` | Weight of the fluid column → mixture density → water cut & gas |
| `lift_dp` | `P_annulus − WHP` | How strongly lift gas is pushed into the tubing |
| `temp_below_res` | `T_res − T_downhole` | More flow carries more reservoir heat up → smaller gap |
| `J_darcy` | `k·h / (141.2·Bo·μ·(ln(re/rw) − 0.75 + s))` | Productivity index from Darcy's law (k, s from PTA) |
| `J_x_pwf` | `J · Pwf` | The Pwf part of the inflow equation `q = J·(P_res − Pwf)` |
| `days` | days since 1 Jan 2025 | Reservoir pressure falls with production (depletion) |
| `on_stream_frac` | `1 − shut-in minutes / 1440` | Daily rate = flowing rate × fraction of day flowing |

Units: rates STB/d (oil, water) and Mscf/d (gas); pressures psia; k md; h, rw, re ft; μ cp; Bo rb/STB.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path('..', 'src').resolve()))
import features as F
import load

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

hf = load.load_high_freq_clean()
tests = load.load_well_tests_clean()
wells = F.build_well_table(load.load_pvt(), load.load_completion(), load.load_own_k_skin())   # OUR build-up k & skin (src/own_well_properties.py)

test_table = F.add_physics_features(F.build_test_table(tests, hf), wells)
daily_table = F.add_physics_features(F.build_daily_table(hf), wells)

test_table.to_parquet(load.PROCESSED / 'test_features.parquet', index=False)
daily_table.to_parquet(load.PROCESSED / 'daily_features.parquet', index=False)
print('test table :', test_table.shape)
print('daily table:', daily_table.shape)
wells.round(2)

## Key finding: the data follows the inflow equation `q_oil = J · (P_res − Pwf)`

If that equation holds, then for every test `P_res = Pwf + q_oil / J`. Using the Darcy J:
- the implied reservoir pressure is **smooth and slowly falling** for every well (depletion), and
- it **matches the pressure reached in the shut-in build-ups** (×), which is an independent measurement.

(Pressures here are at gauge depth.)

In [ ]:
test_table['P_res_implied'] = test_table.dh_pressure_psia + test_table.oil_rate_stbd / test_table.J_darcy

shut_in_max = hf[hf.shut_in].groupby('well_name').agg(time=('timestamp', 'max'), p=('dh_pressure_psia', 'max'))

fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True)
for ax, (well, g) in zip(axes.flat, test_table.groupby('well_name')):
    ax.plot(g.time, g.P_res_implied, 'o-', label='P_res implied by tests')
    if well in shut_in_max.index:
        ax.plot(shut_in_max.loc[well, 'time'], shut_in_max.loc[well, 'p'], 'rx', ms=10, label='shut-in max P')
    ax.set_title(well)
    ax.tick_params(axis='x', rotation=45)
axes[0, 0].legend(fontsize=7)
axes[0, 0].set_ylabel('psia')
fig.tight_layout()
plt.show()

## Which features explain changes *within* a well?
We subtract each well's average first, so the correlation shows what moves the rate of one well over time
(not just that GL-07 is bigger than GL-03).

In [ ]:
feature_cols = ['whp_psia', 'whp_downstream_psia', 'dh_pressure_psia', 'dh_temperature_degf',
                'annulus_pressure_psia', 'gas_lift_rate_mscfd', 'choke_opening_pct', 'choke_dp',
                'choke_p_ratio', 'choke_flow_term', 'gilbert_term', 'tubing_dp', 'tubing_gradient',
                'lift_dp', 'temp_below_res', 'days']
label_cols = ['oil_rate_stbd', 'liquid_rate_stbd', 'water_cut_pct', 'gor_scf_stb']
cols = feature_cols + label_cols

within = test_table[cols] - test_table.groupby('well_name')[cols].transform('mean')
within.corr().loc[feature_cols, label_cols].round(2).sort_values('oil_rate_stbd')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for well, g in test_table.groupby('well_name'):
    axes[0].plot(g.choke_flow_term, g.liquid_rate_stbd, 'o', label=well)
    axes[1].plot(g.days, g.water_cut_pct, 'o-')
    axes[2].plot(g.days, g.gor_scf_stb, 'o-')
axes[0].set(xlabel='choke_flow_term', ylabel='liquid STB/d', title='Liquid vs choke flow term')
axes[1].set(xlabel='days', ylabel='water cut %', title='Water cut rises with time')
axes[2].set(xlabel='days', ylabel='GOR scf/STB', title='GOR per well')
axes[0].legend(fontsize=7)
fig.tight_layout()
plt.show()

## Daily table: shut-in days and missing-data days
- `on_stream_frac` < 1 → partly shut in. 0 → fully shut in (rate = 0).
- `no_data` = True → no sensor data that day at all (outage); features interpolated from neighbouring days.

In [ ]:
daily_table[(daily_table.on_stream_frac < 1) | daily_table.no_data][
    ['well_name', 'date', 'on_stream_frac', 'no_data', 'dh_pressure_psia', 'choke_opening_pct']].round(3)